<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [2]:
raw_data = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [14]:
import numpy as np
import pandas as pd
raw_data.head()


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,engagement_rate
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20.0,0.0,67.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1.0,0.0,0.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125.0,1.0,616.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.008,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7.0,0.0,28.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11.0,0.0,25.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,<NA>


In [43]:
raw_data.loc[raw_data['gsc_data_available']==False, [
 'gsc_impressions',
 'gsc_clicks',
]]=np.nan

raw_data.loc[raw_data['ga4_data_available']==False, [
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec']]=np.nan

raw_data.loc[raw_data['gsc_sum_position']==0, 'gsc_sum_position'] = np.nan
raw_data.loc[raw_data['gsc_avg_position']==0, 'gsc_avg_position'] = np.nan

In [33]:
raw_data['ctr'] = (raw_data['gsc_clicks'] / raw_data['gsc_impressions']) * 100
raw_data['engagement_rate'] = (raw_data['ga4_engaged_sessions'] / raw_data['ga4_sessions']) * 100
raw_data['scroll_rate'] = (raw_data['scroll_events'] / raw_data['ga4_pageviews']) * 100
raw_data['scroll_rate'].isin([np.inf, -np.inf]).sum()
raw_data.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,engagement_rate,scroll_rate
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20.0,0.0,67.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1.0,0.0,0.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125.0,1.0,616.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.8,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7.0,0.0,28.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11.0,0.0,25.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.0,<NA>,<NA>


In [50]:
meta_data = con.sql(f"""
SELECT content_hash_id, competition_level
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
""").df()
raw_data = raw_data.merge(meta_data, on='content_hash_id', how='left')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [58]:
competition_map = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
raw_data['competition_level_encoded'] = raw_data['competition_level'].map(competition_map)

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
"""
1. ctr

Meaning: click-through rate — share of search impressions that resulted in a click, ×100 scale (matches FlyRank's convention)
Missing handling: NaN when gsc_data_available = False (no GSC data at all for this row) — never filled, since fabricating a rate for a page with no search presence would be dishonest
Available when?: known at the decision moment because it's computed from that day's own GSC log — no future data involved

2. gsc_avg_position

Meaning: average search ranking position for that day
Missing handling: NaN when GSC unavailable, OR when the raw value was 0 (a logically impossible position, confirmed via gsc_sum_position cross-check — corrected to null per FlyRank's own documented convention)
Available when?: same-day GSC log, no future leakage

3. engagement_rate

Meaning: share of GA4 sessions that were "engaged" sessions, ×100 scale
Missing handling: NaN when ga4_data_available = False; verified no inf cases (denominator never 0-with-nonzero-numerator in this slice)
Available when?: same-day GA4 log

4. scroll_rate

Meaning: scroll events per pageview, ×100 scale (can exceed 100, matches FlyRank convention)
Missing handling: NaN when GA4 unavailable; the inf edge case (0 pageviews, nonzero scroll events — a GA4 tracking quirk) deliberately set to 0, your judgment call, since "no views but a scroll fired" behaves like near-zero engagement in SEO terms
Available when?: same-day GA4 log

5. competition_level_encoded

Meaning: ordinal encoding of keyword competition intensity (LOW=0, MEDIUM=1, HIGH=2), explicit dictionary mapping (not alphabetical, to preserve real order)
Missing handling: NaN preserved when no keyword data exists for that content item (per dictionary: blank when no keyword context) — not filled, since there's no safe default competition level to assume
Available when?: this is a static content-level attribute from dim_content, not a daily-changing value — known at any decision moment for that content item, since it doesn't depend on the report date
"""

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.